In [1]:
# STANDIN A0
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-For-Cybersecurity-Lab-4.2-Robustness-attacks


train 267,984 rows   attack rate 0.1507
val    89,328 rows   attack rate 0.1507
test   89,329 rows   attack rate 0.1507


68 features: 58 continuous, 10 flag columns
Flag columns: Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count
  tree    loaded from models/tree.joblib
  logreg  loaded from models/logreg.joblib


  forest  loaded from models/forest.joblib


ATTACK_IDX: 50 detected attacks, forest probability 1.000-1.000
  attack types: {'DDoS': 25, 'DoS Hulk': 22, 'DoS GoldenEye': 2, 'SSH-Patator': 1}
EXPLAIN_IDX: 500 random test flows, 68 of them attacks
Setup complete: 20 shared names ready. The models are scored in step A2.


<!-- STEP C1 -->
## Part C: Break the model on purpose

### Step C1: Sort the features and set a budget

An evasion attack is only honest if the attacker changes things a real attacker can change (lab
section 3.4). The attacker sends the **forward** traffic, so they can pad packets and wait longer. They
do not control the victim's replies, and they cannot finish the attack with fewer packets than it
needs. Every feature therefore goes into one of three groups:

| Group | Meaning | Why |
|---|---|---|
| **Free** | timing (inter-arrival times, idle and active timers, flow duration) and forward packet / header / segment sizes | Waiting longer and padding packets cost the attacker almost nothing |
| **Costly** | forward packet and byte counts (`Total Fwd Packets`, `Total Length of Fwd Packets`, forward subflows, forward data packets) | Possible, but every extra packet is more work and makes the attack louder |
| **Fixed** | everything the responder produces (all backward features, its initial window), all flag counts, and every feature we are unsure about | The victim's network stack produces these; the attacker cannot touch them |

The groups are assigned with **name rules in code**, not by hand, in this order: backward or flag →
Fixed; timing → Free; forward sizes → Free; forward counts → Costly; anything else → Fixed. When in
doubt, Fixed: an attack that is weaker than reality is a safer mistake than one that lets the attacker
do the impossible.

Our data has no `Destination Port` or `Protocol` (Lab 1 removed them as ID columns), so those two
Fixed features from the lab text do not appear.

In [2]:
# STEP C1
FREE_SIZES = ["Fwd Header Length", "Avg Fwd Segment Size", "min_seg_size_forward"]
COSTLY_COUNTS = ["Total Fwd Packets", "Total Length of Fwd Packets", "Subflow Fwd Packets",
                 "Subflow Fwd Bytes", "act_data_pkt_fwd"]


def assign_group(name):
    """Free / Costly / Fixed from the feature name, plus the rule that decided it.

    The order matters: the backward check comes first, otherwise 'Bwd IAT Mean' would be Free
    because it contains 'IAT'.
    """
    if "Bwd" in name or "backward" in name.lower():
        return "Fixed", "backward direction: produced by the victim"
    if "Flag" in name:
        return "Fixed", "flag: set by the network stack / responder"
    if any(key in name for key in ("IAT", "Idle", "Active")) or name == "Flow Duration":
        return "Free", "timing: the attacker can wait longer"
    if name.startswith("Fwd Packet Length") or name in FREE_SIZES:
        return "Free", "forward size: the attacker can pad packets"
    if name in COSTLY_COUNTS:
        return "Costly", "forward count: more packets, a louder attack"
    return "Fixed", "unsure (mixes both directions or is derived): Fixed to be safe"


_assigned = {name: assign_group(name) for name in FEATURES}
GROUP = {name: group for name, (group, _) in _assigned.items()}
GROUP_RULE = {name: rule for name, (_, rule) in _assigned.items()}

assert set(GROUP) == set(FEATURES)
counts = pd.Series(GROUP).value_counts().reindex(["Free", "Costly", "Fixed"])
print(counts.to_string())
assert counts.to_dict() == {"Free": 25, "Costly": 5, "Fixed": 38}
for group in ("Free", "Costly", "Fixed"):
    names = [f for f in FEATURES if GROUP[f] == group]
    print(f"\n{group} ({len(names)}):\n  " + ", ".join(names))

Free      25
Costly     5
Fixed     38

Free (25):
  Flow Duration, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Fwd Header Length, Avg Fwd Segment Size, min_seg_size_forward, Active Mean, Active Std, Active Max, Active Min, Idle Mean, Idle Std, Idle Max, Idle Min

Costly (5):
  Total Fwd Packets, Total Length of Fwd Packets, Subflow Fwd Packets, Subflow Fwd Bytes, act_data_pkt_fwd

Fixed (38):
  Total Backward Packets, Total Length of Bwd Packets, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Fwd URG Flags, Bwd Header Length, Fwd Packets/s, Bwd Packets/s, Min Packet Length, Max Packet Length, Packet Length Mean, Packet Length Std, Packet Length Variance, FIN

In [3]:
# STEP C1
# The Fixed features that are Fixed only because we were unsure, with the reason.
unsure = [f for f in FEATURES if GROUP_RULE[f].startswith("unsure")]
print(f"{len(unsure)} features are Fixed by the 'when unsure, Fixed' rule:")
for f in unsure:
    print(f"  {f}")

11 features are Fixed by the 'when unsure, Fixed' rule:
  Flow Bytes/s
  Flow Packets/s
  Fwd Packets/s
  Min Packet Length
  Max Packet Length
  Packet Length Mean
  Packet Length Std
  Packet Length Variance
  Down/Up Ratio
  Average Packet Size
  Init_Win_bytes_forward


<!-- STEP C1 -->
**Why the "unsure" features are Fixed.** They fall in two kinds.

- *Mixed direction*: `Min/Max Packet Length`, `Packet Length Mean/Std/Variance`, `Average Packet Size`
  and `Down/Up Ratio` combine the attacker's packets with the victim's replies. The attacker can move
  only part of them, and not reliably.
- *Derived rates*: `Flow Bytes/s`, `Flow Packets/s` and `Fwd Packets/s` are computed from counts and
  duration. Waiting longer *lowers* them, so the attacker cannot raise them independently.

`Init_Win_bytes_forward` is set by the attacker's own TCP stack and could arguably be Costly. We keep
it Fixed for now; if C2's constrained attack evades nothing, it is the first candidate to move (the lab
allows this, as long as the report says so).

<!-- STEP C1 -->
### The constraint check and the budget

- `can_change(feature, old_value, new_value)` is `True` only when the feature is Free or Costly **and**
  the new value is at least the old one. The attacker can add packets and wait longer, never remove or
  speed up.
- **Budget:** one step raises one feature by at most **half its training standard deviation**, and at
  most **five features** may change in total.

In [4]:
# STEP C1
STEP_SIZE = 0.5 * TRAIN_STD          # pandas Series: the largest single-step increase per feature
MAX_CHANGES = 5                      # at most five different features may change per flow
CHANGEABLE = [f for f in FEATURES if GROUP[f] in ("Free", "Costly")]


def can_change(feature, old_value, new_value):
    """True only if the attacker may move this feature from old_value to new_value."""
    return GROUP[feature] in ("Free", "Costly") and new_value >= old_value


# Tests: no Fixed feature can ever change; nothing can decrease; Free and Costly can increase.
for f in FEATURES:
    if GROUP[f] == "Fixed":
        assert not can_change(f, 0.0, 1.0) and not can_change(f, 5.0, 5.0)
    else:
        assert can_change(f, 0.0, STEP_SIZE[f]) and can_change(f, 3.0, 3.0)
        assert not can_change(f, 1.0, 0.0)
print(f"can_change tests passed: {len(CHANGEABLE)} changeable features, "
      f"{len(FEATURES) - len(CHANGEABLE)} Fixed, none of which can change")

can_change tests passed: 30 changeable features, 38 Fixed, none of which can change


<!-- STEP C1 -->
### What one step means in real units

Part B showed that the standard deviations of our columns are inflated by outliers. The step size is
half a standard deviation, so it is worth seeing what that means for a typical flow before running the
attack: a "small" step can be huge compared with the median.

In [5]:
# STEP C1
def unit(name):
    """Unit of a changeable feature, for reading the table below."""
    if any(key in name for key in ("IAT", "Idle", "Active", "Duration")):
        return "µs"
    if "Length" in name or "Bytes" in name or "Size" in name or "seg_size" in name:
        return "bytes"
    return "packets"


steps = pd.DataFrame({
    "group": [GROUP[f] for f in CHANGEABLE],
    "unit": [unit(f) for f in CHANGEABLE],
    "train_median": TRAIN_MEDIAN[CHANGEABLE],
    "step_size": STEP_SIZE[CHANGEABLE],
}, index=pd.Index(CHANGEABLE, name="feature"))
steps["step_over_median"] = steps["step_size"] / steps["train_median"].abs().replace(0, np.nan)
print(steps.sort_values("step_over_median", ascending=False)
      .to_string(float_format=lambda v: f"{v:,.1f}"))

                              group     unit  train_median    step_size  step_over_median
feature                                                                                  
Fwd IAT Min                    Free       µs           3.0  4,773,842.9       1,591,281.0
Flow IAT Min                   Free       µs           4.0  1,584,409.6         396,102.4
Fwd IAT Total                  Free       µs         710.0 18,396,494.8          25,910.6
Fwd IAT Max                    Free       µs         632.5 13,509,832.8          21,359.4
Fwd IAT Mean                   Free       µs         375.5  5,262,371.2          14,014.3
min_seg_size_forward           Free    bytes          20.0     81,021.8           4,051.1
Fwd Header Length              Free    bytes          64.0     81,388.4           1,271.7
act_data_pkt_fwd             Costly  packets           1.0        344.4             344.4
Flow IAT Std                   Free       µs      17,985.3  4,433,822.0             246.5
Flow Durat

<!-- STEP C1 -->
**What one step means.** The steps are large compared with an ordinary flow, because the standard
deviations are inflated by outliers (B1). Most are still *possible* for a patient attacker: one step
adds 4.8 s to the shortest gap between the attacker's packets (`Fwd IAT Min`), 18 s to the flow
duration, 13 s to the idle timers, or about 360 extra packets (Costly).

Two Free steps are **not physically possible**:

- `min_seg_size_forward` is a header size, between 0 and 60 bytes for every normal flow, but one step
  adds 81,022 bytes. Its std comes almost entirely from a few recording-bug values around −84 million.
- `Fwd Header Length` (total header bytes sent by the attacker) gains 81,388 bytes per step. That
  would take roughly 2,000 extra packets, which is not free.

We keep the lab's budget exactly as written, so the attack follows the lab. In C2 we check which
features the successful evasions actually used; if they rely on these two, the report says so (and C2
can re-run with each new value capped at the training maximum as a sensitivity check). Rows whose
median is 0 (`Idle …`, `Active …`, `Fwd IAT Std`) show no ratio: most flows never go idle, so any
step is "infinitely" larger than the median.

In [6]:
# STEP C1
groups_table = pd.DataFrame({
    "feature": FEATURES,
    "group": [GROUP[f] for f in FEATURES],
    "rule": [GROUP_RULE[f] for f in FEATURES],
    "step_size": [STEP_SIZE[f] if GROUP[f] != "Fixed" else np.nan for f in FEATURES],
    "train_median": TRAIN_MEDIAN[FEATURES].to_numpy(),
})
groups_table.to_csv("results/tables/C1_feature_groups.csv", index=False, float_format="%.4g")
print("Saved results/tables/C1_feature_groups.csv")

Saved results/tables/C1_feature_groups.csv
